# Reflection Pattern

The first pattern we are going to implement is the **reflection pattern**. 

---

<img src="../img/reflection_pattern.png" alt="Alt text" width="600"/>

---

This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

## Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? Python code?

For this example, I've decided to test the Python coding skills of Llama3 70B (that's the LLM we are going to use for all the tutorials). In particular, we are going to ask our LLM to code a famous sorting algorithm: **Merge Sort**. 

---

<img src="../img/mergesort.png" alt="Alt text" width="500"/>

### Groq Client and relevant imports

In [7]:
import os
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()
GROQ_MODEL = "openai/gpt-oss-120b"
client = Groq()

We will start the **"generation"** chat history with the system prompt, as we said before. In this case, let the LLM act like a Python 
programmer eager to receive feedback / critique by the user.

In [2]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

Now, as the user, we are going to ask the LLM to generate an implementation of the **Merge Sort** algorithm. Just add a new message with the **user** role to the chat history.

In [3]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Merge Sort algorithm"
    }
)

Let's generate the first version of the essay.

In [ ]:
mergesort_code = client.chat.completions.create(
    messages = generation_chat_history,
    model = GROQ_MODEL
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

In [8]:
display_markdown(mergesort_code, raw=True)

Here is a clean, well‑documented Python implementation of **Merge Sort**.  
The implementation works for any sequence that can be indexed (e.g., lists, tuples) and returns a **new sorted list** without modifying the original data.  
I also include a small test harness that demonstrates usage and verifies correctness with Python’s built‑in `sorted` as a reference.

```python
"""
merge_sort.py

A straightforward, type‑annotated implementation of the Merge Sort
algorithm in pure Python.

Features
--------
* Works with any sortable elements (numbers, strings, custom objects with
  __lt__ defined, etc.).
* Returns a **new** sorted list – the original input remains unchanged.
* Uses a top‑down recursive approach with a helper function that merges
  two already‑sorted sub‑lists.
* Includes a simple test harness that can be run with `python -m merge_sort`
  or imported and used as a library.
"""

from __future__ import annotations

from typing import List, Sequence, TypeVar

# Generic type variable that must support the < operator
T = TypeVar("T")


def merge(left: List[T], right: List[T]) -> List[T]:
    """
    Merge two sorted lists into a single sorted list.

    Parameters
    ----------
    left, right : List[T]
        The two input lists, each already sorted in non‑decreasing order.

    Returns
    -------
    List[T]
        A new list containing all elements from *left* and *right*,
        sorted.
    """
    merged: List[T] = []
    i = j = 0

    # Walk through both lists, picking the smallest remaining element each time.
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # One of the lists may have leftovers – extend them directly.
    if i < len(left):
        merged.extend(left[i:])
    if j < len(right):
        merged.extend(right[j:])

    return merged


def merge_sort(seq: Sequence[T]) -> List[T]:
    """
    Perform a top‑down merge sort on the given sequence.

    Parameters
    ----------
    seq : Sequence[T]
        The input data to sort. It can be a list, tuple, or any sequence type.

    Returns
    -------
    List[T]
        A new list containing the elements of *seq* in sorted order.

    Notes
    -----
    * The algorithm runs in O(n log n) time and O(n) auxiliary space.
    * This implementation is recursive; for very deep recursion you may
      need to increase the recursion limit (unlikely for typical data sizes).
    """
    # Base case: a sequence of length 0 or 1 is already sorted.
    if len(seq) <= 1:
        return list(seq)  # return a mutable copy

    mid = len(seq) // 2
    # Recursively sort the left and right halves
    left_sorted = merge_sort(seq[:mid])
    right_sorted = merge_sort(seq[mid:])

    # Merge the two sorted halves
    return merge(left_sorted, right_sorted)


# --------------------------------------------------------------------------- #
# Optional: simple test harness
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    import random
    import sys

    # Small sanity check against Python's built‑in sorting
    def _test_random_cases(num_cases: int = 100, max_len: int = 20, max_val: int = 100):
        for _ in range(num_cases):
            length = random.randint(0, max_len)
            data = [random.randint(-max_val, max_val) for _ in range(length)]
            expected = sorted(data)
            result = merge_sort(data)
            assert result == expected, f"Failed on {data!r}: got {result!r}, expected {expected!r}"
        print(f"All {num_cases} random tests passed.")

    # Demonstrate usage with a user‑friendly example
    example = [38, 27, 43, 3, 9, 82, 10]
    print("Original :", example)
    print("Sorted   :", merge_sort(example))

    # Run the automated test suite when invoked directly
    _test_random_cases()
```

### How it works
1. **`merge_sort`** splits the input sequence in half recursively until it reaches sub‑sequences of length `0` or `1`.  
2. The **`merge`** helper combines two already‑sorted lists by repeatedly taking the smaller of the two front elements.  
3. The recursion unwinds, producing a fully sorted list.

Feel free to copy the file into your project, import `merge_sort`, and call it on any iterable you need to sort!

## Reflection Step

Now, let's allow the LLM to reflect on its outputs by defining another system prompt. This system prompt will tell the LLM to act as Andrej Karpathy, computer scientist and Deep Learning wizard.

>To be honest, I don't think the fact of acting like Andrej Karpathy will influence the LLM outputs, but it was fun :)

<img src="../img/karpathy.png" alt="Alt text" width="500"/>

In [9]:
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are Andrej Karpathy, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

The user message, in this case,  is the essay generated in the previous step. We simply add the `mergesort_code` to the `reflection_chat_history`.

In [10]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": mergesort_code
    }
)

Now, let's generate a critique to the Python code.

In [12]:
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model=GROQ_MODEL
).choices[0].message.content

In [13]:
display_markdown(critique, raw=True)

Below is a detailed, “Karpathy‑style” review of the merge‑sort code you posted.  
I’ll walk through the design, correctness, performance, style, and testing aspects, then give concrete recommendations you can copy‑‑paste into the file if you like.

---  
## 1. Functional Correctness  

| ✅ | Observation |
|---|--------------|
| **Base case** | `len(seq) <= 1` correctly returns a mutable copy (`list(seq)`). |
| **Merge** | The classic two‑pointer merge works for any type that implements `<=`. |
| **Round‑trip** | The test harness checks many random cases against `sorted`, so you already have a sanity check. |
| **Stability** | Because you use `<=` when picking from the left list, the algorithm is *stable* (equal elements keep their original order). |

**No functional bugs** were found in the core algorithm.

---  
## 2. Complexity & Memory Footprint  

| Metric | Current implementation | Remarks |
|--------|------------------------|---------|
| **Time** | `O(n log n)` – optimal for comparison‑based sorting. |
| **Auxiliary space** | `O(n log n)` worst‑case because each recursive call creates new slices (`seq[:mid]`, `seq[mid:]`) **and** `merge` builds a fresh list. In practice you allocate about `2n` total (one for the final result, one for temporary halves) plus the recursion stack `O(log n)`. |
| **Recursion depth** | `≈ log₂ n`. Python’s default recursion limit (≈1000) is safe up to ~2³⁰ elements, but it can be hit for pathological data (e.g., a list of length 10⁶ on a machine with a low limit). |

If you need to sort huge arrays you might want to:

* **Avoid slicing** – pass start/end indices instead of creating new lists.  
* **Iterative bottom‑up merge sort** – eliminates recursion and can reuse a single auxiliary buffer.  
* **In‑place merge** – trickier but reduces the extra `O(n)` allocation.

---  
## 3. API & Type Safety  

### 3.1. Generic type bound  

```python
T = TypeVar("T", bound="SupportsLessThan")
```

You could tighten the bound to something that explicitly requires `<`/`<=`. The standard library provides `typing.Protocol` for that:

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class SupportsLessThan(Protocol):
    def __lt__(self, other: Any) -> bool: ...

T = TypeVar("T", bound=SupportsLessThan)
```

Now static type‑checkers will complain if you try to sort objects that don’t implement comparison.

### 3.2. Returning the same concrete type  

Right now `merge_sort` always returns a `list`, even if the caller passed a tuple. That’s fine for most use‑cases, but you can make the function more generic:

```python
from collections.abc import Iterable
def merge_sort(seq: Iterable[T]) -> List[T]: …
```

If you really want to preserve the input type, you could add an optional `factory` argument:

```python
def merge_sort(seq: Sequence[T], *, factory: Callable[[Iterable[T]], S] = list) -> S:
    …
    return factory(merged)
```

That’s a bit more advanced, but it shows how to make the API flexible.

### 3.3. `key` and `reverse` arguments  

`sorted` supports `key=` and `reverse=`; adding them makes your function a drop‑in replacement for most quick scripts.

```python
def merge_sort(
    seq: Sequence[T],
    *,
    key: Callable[[T], Any] | None = None,
    reverse: bool = False,
) -> List[T]:
    …
    # Inside merge:
    left_val = key(left[i]) if key else left[i]
    right_val = key(right[j]) if key else right[j]
    if (left_val < right_val) ^ reverse:   # XOR flips direction when reverse=True
        …
```

It adds a few lines, but the payoff in usability is huge.

---  
## 4. Code Style & Idiomatic Python  

| Line | Suggestion |
|------|------------|
| `merged: List[T] = []` | Good use of type hint. |
| `i = j = 0` | Fine. |
| `while i < len(left) and j < len(right):` | Consider using `while i < len(left) and j < len(right):` (no change) – already idiomatic. |
| `if left[i] <= right[j]:` | Using `<=` guarantees stability; keep it. |
| `if i < len(left): merged.extend(left[i:])` | You could combine the two `if`s with `merged.extend(left[i:])` followed by `merged.extend(right[j:])`. Extending an empty slice is a cheap no‑op, so you can drop the `if`s entirely. |
| `return list(seq)` in the base case | Perfect – makes the return type always mutable. |
| Recursion limit comment | Nice. You might also expose a helper to let the user bump the limit if they really need to (`sys.setrecursionlimit`). |
| Test harness in `__main__` | Good. For a real library you could replace it with `unittest`/`pytest` fixtures. |
| `import sys` – never used | Remove it (or use it for the recursion‑limit note). |
| `random` import inside `__main__` | Fine; keeps the import cost out of the library path. |
| `def _test_random_cases(...):` | Private helper – fine, but consider using `typing.NoReturn` for the return type if you want to be explicit that it never returns a value (it raises on failure). |

### Minor stylistic polish  

```python
def merge(left: List[T], right: List[T]) -> List[T]:
    merged: List[T] = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Extending empty slices is cheap, so we can drop the guards.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged
```

---  
## 5. Testing Improvements  

* **Property‑based testing** – `hypothesis` can generate massive edge cases (duplicate elements, custom objects, etc.) with minimal code.
* **Performance benchmark** – a quick `timeit` comparison against `sorted` helps spot pathological cases (e.g., already sorted, reverse sorted).  
* **Edge‑case coverage** – empty list, single element, all equal elements, already sorted, reverse sorted, large random list (~10⁶ items) (just to sanity‑check memory usage).

Example using `pytest` + `hypothesis`:

```python
from hypothesis import given, strategies as st

@given(st.lists(st.integers()))
def test_merge_sort_agrees_with_builtin(data):
    assert merge_sort(data) == sorted(data)
```

Add this to a `tests/` directory; CI will run it automatically.

---  
## 6. Optional Advanced Features  

| Feature | Why it’s useful | How to add (brief) |
|---------|----------------|--------------------|
| **Bottom‑up iterative version** | Eliminates recursion depth limits; often faster due to better cache locality. | Write a loop that merges runs of size `width = 1, 2, 4, …` until `width >= n`. |
| **In‑place merge using `deque` or `bisect`** | Saves the extra `O(n)` buffer, useful for memory‑constrained environments. | More complex; see “In‑place merge sort” (e.g., `timsort` does this). |
| **Parallel merge sort** | Takes advantage of multi‑core CPUs for large data sets. | Use `concurrent.futures.ProcessPoolExecutor` to sort halves concurrently, then merge. |
| **Typed NumPy support** | If you ever need to sort numeric arrays, `numpy.sort` is dramatically faster. | Not needed for pure‑Python lists, but worth mentioning in docs. |

---  
## 7. Revised File (copy‑paste ready)  

Below is a cleaned‑up version that incorporates the most impactful suggestions (type‑bound, key/reverse, simplified `merge`, removed unused import, doc‑string tweaks, and a tiny `unittest` stub). Feel free to cherry‑pick parts you like.

```python
"""
merge_sort.py

A type‑annotated, stable, top‑down merge‑sort implementation.

Features
--------
* Works with any sortable elements (numbers, strings, custom objects with __lt__).
* Returns a **new** list; the original input is unchanged.
* Optional ``key`` and ``reverse`` parameters mimic the built‑in ``sorted``.
* Includes a simple test harness that can be executed with ``python -m merge_sort``.
"""

from __future__ import annotations

from typing import Any, Callable, List, Protocol, Sequence, TypeVar

# --------------------------------------------------------------------------- #
#  Type definitions
# --------------------------------------------------------------------------- #
class SupportsLessThan(Protocol):
    """Protocol for objects supporting the ``<`` operator."""
    def __lt__(self, other: Any) -> bool: ...

T = TypeVar("T", bound=SupportsLessThan)


# --------------------------------------------------------------------------- #
#  Core algorithm
# --------------------------------------------------------------------------- #
def _merge(
    left: List[T],
    right: List[T],
    *,
    key: Callable[[T], Any] | None = None,
    reverse: bool = False,
) -> List[T]:
    """Merge two sorted lists into a new sorted list."""
    merged: List[T] = []
    i = j = 0

    while i < len(left) and j < len(right):
        l_val = key(left[i]) if key else left[i]
        r_val = key(right[j]) if key else right[j]

        # ``reverse`` flips the comparison direction.
        if (l_val < r_val) ^ reverse:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Extending with an empty slice is a cheap no‑op.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort(
    seq: Sequence[T],
    *,
    key: Callable[[T], Any] | None = None,
    reverse: bool = False,
) -> List[T]:
    """
    Recursively sort *seq* using the merge‑sort algorithm.

    Parameters
    ----------
    seq : Sequence[T]
        Input data – any sequence (list, tuple, etc.).
    key : Callable[[T], Any] | None, optional
        Function that extracts a comparison key from each element (like ``sorted``).
    reverse : bool, optional
        If ``True``, sort in descending order.

    Returns
    -------
    List[T]
        A new list containing the elements of *seq* in sorted order.
    """
    if len(seq) <= 1:
        return list(seq)   # always return a mutable copy

    mid = len(seq) // 2
    left_sorted = merge_sort(seq[:mid], key=key, reverse=reverse)
    right_sorted = merge_sort(seq[mid:], key=key, reverse=reverse)

    return _merge(left_sorted, right_sorted, key=key, reverse=reverse)


# --------------------------------------------------------------------------- #
#  Test harness
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    import random
    import sys
    import unittest

    class MergeSortTests(unittest.TestCase):
        def test_random(self):
            for _ in range(200):
                data = [random.randint(-1000, 1000) for _ in range(random.randint(0, 50))]
                self.assertEqual(merge_sort(data), sorted(data))

        def test_key(self):
            data = ["apple", "Banana", "cherry", "date"]
            self.assertEqual(
                merge_sort(data, key=str.lower),
                sorted(data, key=str.lower),
            )

        def test_reverse(self):
            data = list(range(10))
            self.assertEqual(merge_sort(data, reverse=True), list(reversed(data)))

    # Example usage when run as a script
    example = [38, 27, 43, 3, 9, 82, 10]
    print("Original :", example)
    print("Sorted   :", merge_sort(example))

    # Run the unittest suite
    unittest.main(argv=[sys.argv[0]], exit=False)
```

### What changed?

* **`SupportsLessThan` protocol** – gives static type checkers more information.  
* **`key` / `reverse`** – mirrors the signature of `sorted`.  
* **`_merge`** – now respects `key`/`reverse` and has a tiny simplification.  
* **Removed unused `sys` import** (still used for `unittest.main`).  
* Swapped the ad‑hoc assert‑loop for a proper `unittest` class – easier to extend and integrates with CI pipelines.  
* Added a

Finally, we just need to add this *critique* to the `generation_chat_history`, in this case, as the `user` role.

In [14]:
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

## Generation Step (II)

In [15]:
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model=GROQ_MODEL
).choices[0].message.content

In [16]:
display_markdown(essay, raw=True)

Below is a **drop‑in replacement** for the file you posted.  
It incorporates the most useful suggestions from the review:

* Uses a **`SupportsLessThan` protocol** so static type‑checkers know the elements are comparable.  
* Adds the **`key`** and **`reverse`** arguments (just like the built‑in `sorted`).  
* Simplifies the `merge` helper while still being **stable**.  
* Removes the unused import and replaces the ad‑hoc test loop with a tiny **`unittest`** suite (easy to run in CI).  
* Keeps the original public API (`merge_sort(seq) -> List`) but now returns a list that respects the optional arguments.

You can copy‑paste the whole block into `merge_sort.py` (or whatever module name you prefer).

```python
"""
merge_sort.py

A type‑annotated, stable, top‑down merge‑sort implementation.

Features
--------
* Works with any sortable elements (numbers, strings, custom objects with __lt__).
* Returns a **new** list; the original input is unchanged.
* Optional ``key`` and ``reverse`` parameters mimic the built‑in ``sorted``.
* Includes a simple test harness that can be executed with ``python -m merge_sort``.
"""

from __future__ import annotations

from typing import Any, Callable, List, Protocol, Sequence, TypeVar

# --------------------------------------------------------------------------- #
#  Type definitions
# --------------------------------------------------------------------------- #
class SupportsLessThan(Protocol):
    """Protocol for objects supporting the ``<`` operator."""
    def __lt__(self, other: Any) -> bool: ...

T = TypeVar("T", bound=SupportsLessThan)


# --------------------------------------------------------------------------- #
#  Core algorithm
# --------------------------------------------------------------------------- #
def _merge(
    left: List[T],
    right: List[T],
    *,
    key: Callable[[T], Any] | None = None,
    reverse: bool = False,
) -> List[T]:
    """Merge two sorted lists into a new sorted list."""
    merged: List[T] = []
    i = j = 0

    while i < len(left) and j < len(right):
        l_val = key(left[i]) if key else left[i]
        r_val = key(right[j]) if key else right[j]

        # ``reverse`` flips the comparison direction.
        if (l_val < r_val) ^ reverse:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Extending with an empty slice is a cheap no‑op.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort(
    seq: Sequence[T],
    *,
    key: Callable[[T], Any] | None = None,
    reverse: bool = False,
) -> List[T]:
    """
    Recursively sort *seq* using the merge‑sort algorithm.

    Parameters
    ----------
    seq : Sequence[T]
        Input data – any sequence (list, tuple, etc.).
    key : Callable[[T], Any] | None, optional
        Function that extracts a comparison key from each element (like ``sorted``).
    reverse : bool, optional
        If ``True``, sort in descending order.

    Returns
    -------
    List[T]
        A new list containing the elements of *seq* in sorted order.
    """
    # Base case – a sequence of length 0 or 1 is already sorted.
    if len(seq) <= 1:
        return list(seq)   # always return a mutable copy

    mid = len(seq) // 2
    left_sorted = merge_sort(seq[:mid], key=key, reverse=reverse)
    right_sorted = merge_sort(seq[mid:], key=key, reverse=reverse)

    return _merge(left_sorted, right_sorted, key=key, reverse=reverse)


# --------------------------------------------------------------------------- #
#  Test harness
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    import random
    import sys
    import unittest

    class MergeSortTests(unittest.TestCase):
        """A few sanity‑checks that also serve as an example of usage."""

        def test_random_cases(self):
            """Compare against the built‑in sorted on many random inputs."""
            for _ in range(200):
                data = [random.randint(-1000, 1000) for _ in range(random.randint(0, 50))]
                self.assertEqual(merge_sort(data), sorted(data))

        def test_key_argument(self):
            data = ["apple", "Banana", "cherry", "date"]
            self.assertEqual(
                merge_sort(data, key=str.lower),
                sorted(data, key=str.lower),
            )

        def test_reverse_argument(self):
            data = list(range(10))
            self.assertEqual(merge_sort(data, reverse=True), list(reversed(data)))

    # Example usage when run as a script
    example = [38, 27, 43, 3, 9, 82, 10]
    print("Original :", example)
    print("Sorted   :", merge_sort(example))

    # Run the unittest suite (exit=False lets the script continue after tests)
    unittest.main(argv=[sys.argv[0]], exit=False)
```

### How to use it

```python
from merge_sort import merge_sort

# Simple sort
sorted_list = merge_sort([5, 2, 9, 1])

# With a key function (case‑insensitive string sort)
sorted_strings = merge_sort(['b', 'A', 'c'], key=str.lower)

# Descending order
desc = merge_sort([1, 2, 3, 4], reverse=True)
```

The module can also be executed directly (`python -m merge_sort`) to see the example output and run the built‑in tests. Feel free to cherry‑pick any of the additions (type protocol, `key`/`reverse`, the `unittest` block, etc.) that suit your project. Happy coding!

## And the iteration starts again ...

After **Generation Step (II)** the corrected Python code will be received, once again, by Karpathy. Then, the LLM will reflect on the corrected output, suggesting further improvements and the loop will go, over and over for a number **n** of total iterations.

> There's another possibility. Suppose the Reflection step can't find any further improvement. In this case, we can tell the LLM to output some stop string, like "OK" or "Good" that means the process can be stopped. However, we are going to follow the first approach, that is, iterating for a fixed number of times.

## Implementing a class 

Now that you understand the underlying loop of the Reflection Agent, let's implement this agent as a class.

In [1]:
from agentic_patterns import ReflectionAgent

In [2]:
agent = ReflectionAgent()

In [4]:
generation_system_prompt = "You are a Python programmer tasked with generating high quality Python code"

reflection_system_prompt = "You are Andrej Karpathy, an experienced computer scientist"

user_msg = "Generate a Python implementation of the Merge Sort algorithm"

In [5]:
final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=10,
    verbose=1,
)


STEP 1/10

 

GENERATION

 ```python
def merge_sort(arr):
    """Return a new list containing the elements of *arr* sorted in ascending order."""
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    # Merge the two sorted halves
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Append any remaining items
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


# Example usage
if __name__ == "__main__":
    unsorted = [38, 27, 43, 3, 9, 82, 10]
    sorted_list = merge_sort(unsorted)
    print("Unsorted:", unsorted)
    print("Sorted:  ", sorted_list)
```

*Key points*  
- Recursive divide‑and‑conquer approach (O(n log n) time, O(n) auxiliary space).  
- Returns a new list; original input remains unch

## Final result

In [8]:
display_markdown(final_response, raw=True)

```python
from __future__ import annotations

import array
import heapq
import itertools
import random
from typing import (
    Any,
    Callable,
    Generator,
    Iterable,
    Iterator,
    List,
    MutableSequence,
    Optional,
    Sequence,
    TypeVar,
)

T = TypeVar("T")
KeyFunc = Callable[[T], Any] | None


def _compare(a: T, b: T, *, key: KeyFunc = None, reverse: bool = False) -> bool:
    """True if *a* should be placed before *b* (stable).

    Forward: a <= b; Reverse: a >= b.
    """
    ka = key(a) if key else a
    kb = key(b) if key else b
    return ka <= kb if not reverse else ka >= kb


def _merge(
    src: Sequence[T],
    dst: MutableSequence[Optional[T]],
    start: int,
    mid: int,
    end: int,
    *,
    key: KeyFunc = None,
    reverse: bool = False,
) -> None:
    """Merge sorted slices ``src[start:mid]`` and ``src[mid:end]`` into ``dst``."""
    i, j, k = start, mid, start
    while i < mid and j < end:
        if _compare(src[i], src[j], key=key, reverse=reverse):
            dst[k] = src[i]
            i += 1
        else:
            dst[k] = src[j]
            j += 1
        k += 1
    while i < mid:
        dst[k] = src[i]
        i += 1
        k += 1
    while j < end:
        dst[k] = src[j]
        j += 1
        k += 1


def merge_sort(
    arr: Sequence[T],
    *,
    key: KeyFunc = None,
    reverse: bool = False,
) -> List[T]:
    """Return a new list with *arr* sorted. Stable sort (equal elements retain original order)."""
    n = len(arr)
    if n == 0:
        return []

    src: List[T] = list(arr)
    dst: MutableSequence[Optional[T]] = [None] * n

    width = 1
    while width < n:
        for start in range(0, n, 2 * width):
            mid = min(start + width, n)
            end = min(start + 2 * width, n)
            _merge(src, dst, start, mid, end, key=key, reverse=reverse)
        src, dst = dst, src  # type: ignore[assignment]
        width <<= 1
    return src  # type: ignore[return-value]


def merge_sort_inplace(
    arr: MutableSequence[T],
    *,
    buffer: MutableSequence[Optional[T]] | None = None,
    key: KeyFunc = None,
    reverse: bool = False,
    copy_back: bool = True,
) -> None:
    """
    Sort *arr* in place using a stable bottom‑up merge sort.
    Stable sort (equal elements retain original order).
    """
    n = len(arr)
    if n == 0:
        return

    if buffer is None:
        buffer = [None] * n  # type: ignore[assignment]
    elif len(buffer) != n:
        raise ValueError("buffer length must equal input length")

    width = 1
    src, dst = arr, buffer
    while width < n:
        for start in range(0, n, 2 * width):
            mid = min(start + width, n)
            end = min(start + 2 * width, n)
            _merge(src, dst, start, mid, end, key=key, reverse=reverse)
        src, dst = dst, src
        width <<= 1

    if copy_back and src is not arr:
        arr[:] = src[:]  # copy final result back


def merge_sort_iter(
    iterable: Iterable[T],
    *,
    key: KeyFunc = None,
    reverse: bool = False,
) -> Iterator[T]:
    """Yield the sorted elements of *iterable* one by one. Stable sort."""
    return iter(merge_sort(list(iterable), key=key, reverse=reverse))


def merge_chunks(
    *chunks: Iterable[T],
    key: KeyFunc = None,
    reverse: bool = False,
) -> Iterator[T]:
    """Merge already‑sorted iterables lazily. Stable sort."""
    return heapq.merge(*chunks, key=key, reverse=reverse)


def _merge_iter(a: Iterator[T], b: Iterator[T]) -> Iterator[T]:
    """Low‑level stable merge of two sorted iterators."""
    a_next = next(a, None)
    b_next = next(b, None)
    while a_next is not None or b_next is not None:
        if b_next is None or (a_next is not None and a_next <= b_next):
            yield a_next  # type: ignore[misc]
            a_next = next(a, None)
        else:
            yield b_next  # type: ignore[misc]
            b_next = next(b, None)


# ==================== Tests ====================
if __name__ == "__main__":
    # basic correctness
    assert merge_sort([]) == []
    assert merge_sort([1]) == [1]
    assert merge_sort([2, 1, 3, 1, 2]) == [1, 1, 2, 2, 3]

    # key / reverse
    data = [{"v": 5}, {"v": 2}, {"v": 5}]
    assert merge_sort(data, key=lambda d: d["v"]) == [{"v": 2}, {"v": 5}, {"v": 5}]
    assert merge_sort(data, key=lambda d: d["v"], reverse=True) == [
        {"v": 5},
        {"v": 5},
        {"v": 2},
    ]

    # in‑place version
    for lst in ([3, 1, 2], [1], [], [1, 2, 3], [3, 2, 1]):
        orig = lst[:]
        merge_sort_inplace(lst)
        assert lst == sorted(orig)

    # in‑place with custom buffer (list, array, numpy‑like)
    a = [4, 2, 3, 1]
    buf = [None] * len(a)  # type: ignore[assignment]
    merge_sort_inplace(a, buffer=buf)
    assert a == [1, 2, 3, 4]

    arr = array.array("i", [9, 7, 5, 3, 1])
    buf_arr = array.array("i", [0] * len(arr))
    merge_sort_inplace(arr, buffer=buf_arr)
    assert list(arr) == [1, 3, 5, 7, 9]

    # iterator version
    it = merge_sort_iter([5, 3, 4, 1, 2])
    assert list(it) == [1, 2, 3, 4, 5]

    # merge_chunks basics
    merged = list(merge_chunks([1, 4, 7], [2, 5, 8], [3, 6, 9]))
    assert merged == list(range(1, 10))

    # merge_chunks with empty and single iterables
    assert list(merge_chunks()) == []
    assert list(merge_chunks([1, 2, 3])) == [1, 2, 3]

    # merge_chunks with duplicate values and key
    dup_chunks = ([("a", 2), ("b", 1)], [("c", 2), ("d", 0)])
    merged_key = list(merge_chunks(*dup_chunks, key=lambda x: x[1]))
    assert merged_key == [("d", 0), ("b", 1), ("a", 2), ("c", 2)]

    # large random list (10⁵ elements)
    large = [random.randint(0, 1_000_000) for _ in range(100_000)]
    assert merge_sort(large) == sorted(large)

    # edge‑case tests
    for lst in ([], [42], [1, 2, 3], [3, 2, 1]):
        assert merge_sort(lst) == sorted(lst)
        tmp = lst[:]
        merge_sort_inplace(tmp)
        assert tmp == sorted(lst)

    # duplicate keys with key function
    dup = ["apple", "apricot", "banana", "avocado"]
    assert merge_sort(dup, key=lambda s: s[0]) == ["apple", "apricot", "avocado", "banana"]

    print("All tests passed.")
```